In [1]:
!pip install pymc

  Using cached pymc-6.4.0-py3-none-any.whl.metadata (17 kB)
  Using cached arviz-1.3.0-py3-none-any.whl.metadata (7.9 kB)
  Using cached better_optimize-0.5.1-py3-none-any.whl.metadata (18 kB)
  Using cached cachetools-6.2.6-py3-none-any.whl.metadata (5.6 kB)
  Using cached pytensor-3.3.8-cp312-cp312-win_amd64.whl.metadata (7.3 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached threadpoolctl-3.8.0-py3-none-any.whl.metadata (29 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached numba-0.68.0-cp312-cp312-win_amd64.whl.metadata (2.9 kB)
  Using cached llvmlite-0.50.0-cp312-cp312-win_amd64.whl.metadata (5.2 kB)
  Using cached markdown_it_py-4.3.0-py3-none-any.whl.metadata (7.9 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
Using cached pymc-6.4.0-py3-none-any.whl (619 kB)
Using cached arviz-1.3.0-py3-none-any.whl (9.2 kB)
Using cached better_optimize-0.5.1-py3-none-any.whl (77 kB)
Using cached cachetools-6.2.6-py

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, beta

print("--- 1. Random-walk Metropolis-Hastings for Standard Normal ---")
def target_pdf(x): return norm.pdf(x, 0, 1)

def rw_mh(n_samples=10000, proposal_std=1.0):
    samples = np.zeros(n_samples)
    samples[0] = 0.0
    for i in range(1, n_samples):
        current = samples[i-1]
        proposal = np.random.normal(current, proposal_std)
        # Acceptance ratio
        acceptance = target_pdf(proposal) / target_pdf(current)
        if np.random.rand() < acceptance:
            samples[i] = proposal
        else:
            samples[i] = current
    return samples

mh_samples = rw_mh()
print(f"Mean of MH samples: {np.mean(mh_samples):.4f} (Target is 0)")

print("\n--- 2. Bernoulli with Beta(2,2) prior (8 successes in 12 trials) ---")
# Posterior is Beta(alpha + successes, beta + failures)
alpha_post = 2 + 8
beta_post = 2 + 4
print(f"Posterior Distribution: Beta({alpha_post}, {beta_post})")
post_mean = alpha_post / (alpha_post + beta_post)
print(f"Posterior Mean: {post_mean:.4f}")

print("\n--- 4. PyMC Model (Run this if PyMC is installed) ---")
try:
    import pymc as pm
    # Generate some dummy normal data
    true_data = np.random.normal(10, 2, size=100)
    
    with pm.Model() as model:
        # Priors
        mu = pm.Normal('mu', mu=0, sigma=10)
        sigma = pm.HalfNormal('sigma', sigma=5)
        # Likelihood
        obs = pm.Normal('obs', mu=mu, sigma=sigma, observed=true_data)
        # Sample
        trace = pm.sample(500, tune=500, cores=1, chains=1, progressbar=False)
    print("PyMC Model executed successfully.")
except ImportError:
    print("PyMC not installed. Run 'pip install pymc' to execute this part.")

--- 1. Random-walk Metropolis-Hastings for Standard Normal ---
Mean of MH samples: 0.0201 (Target is 0)

--- 2. Bernoulli with Beta(2,2) prior (8 successes in 12 trials) ---
Posterior Distribution: Beta(10, 6)
Posterior Mean: 0.6250

--- 4. PyMC Model (Run this if PyMC is installed) ---


g++ not available, if using conda: `conda install gxx`
Initializing NUTS using jitter+adapt_diag...
Sequential sampling (1 chains in 1 job)
NUTS: [mu, sigma]
C:\Users\NGOSA\.conda\envs\ICD341\Lib\site-packages\pymc\step_methods\hmc\quadpotential.py:321: RuntimeWarning: overflow encountered in dot
  return 0.5 * np.dot(x, v_out)
Sampling 1 chain for 500 tune and 500 draw iterations (500 + 500 draws total) took 1 seconds.
Only one chain was sampled, this makes it impossible to run some convergence checks


PyMC Model executed successfully.


3. Explain trace plots, ACF, R-hat, and ESS:
• Trace Plot: A line plot of the sampled values over iterations. It should look like a "hairy caterpillar" (stationary, well-mixed). If it drifts or gets stuck, the chain hasn't converged.
• ACF (Autocorrelation Function): Measures how correlated a sample is with its past samples. It should drop to zero quickly. Slow decay means the sampler is taking small steps and struggling to explore the space efficiently.
• R-hat (Gelman-Rubin statistic): Compares the variance between multiple independent chains to the variance within the chains. Values > 1.01 indicate a lack of convergence.
• ESS (Effective Sample Size): Because MCMC samples are autocorrelated, 10,000 samples might only contain 1,000 "independent" pieces of information. ESS estimates this true amount of information.